# TP1 - Du texte brut au modele bigramme

Ce notebook transforme Tiny Shakespeare en exemples supervises, entraine une table de logits et compare une generation avant/apres apprentissage.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
DATA_PATH = Path('input.txt')
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not DATA_PATH.exists():
    urlretrieve(URL, DATA_PATH)
text = DATA_PATH.read_text(encoding='utf-8')
print(f'Corpus: {len(text):,} caracteres')
print(repr(text[:300]))

Un token est ici un caractere. L'encodage UTF-8 permet de decoder correctement les octets en texte Unicode. Le corpus est une longue sequence, pas une liste d'exemples deja separes.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)
message = 'hii there'
print('vocab_size =', vocab_size)
print(encode(message))
print(decode(encode(message)))
assert decode(encode(message)) == message

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
block_size, batch_size = 8, 32
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    print(x[:t + 1].tolist(), '->', y[t].item())

def get_batch(split):
    source = train_data if split == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    xb = torch.stack([source[i:i + block_size] for i in starts])
    yb = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return xb, yb

xb, yb = get_batch('train')
print('batch shapes:', xb.shape, yb.shape)

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.reshape(B * T, C), targets.reshape(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context, 200)[0].tolist()))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
losses = []
for step in range(1000):
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step in (0, 99, 999):
        print(f'step {step + 1}: loss={loss.item():.4f}')

print(decode(model.generate(context, 500)[0].tolist()))

### Bilan

- `B`, `T`, `C` designent batch, longueur de contexte et nombre de classes.
- `x` et `y` sont decales d'un caractere pour apprendre le prochain token.
- Pendant la generation, seul le dernier pas temporel est utilise car c'est la prochaine distribution qui nous interesse.
- Aplatir `(B,T,C)` en `(B*T,C)` permet de calculer une loss pour toutes les positions en parallele.
- L'apprentissage modifie chaque ligne de la table: elle devient une distribution de caracteres plausibles apres le caractere d'entree.
